## 11. Survival Analysis — Kaplan-Meier

Survival analysis models *when* an event occurs, not just *whether* it does.

- **Kaplan-Meier:** Non-parametric — shows probability of surviving (not defaulting) to each month.
- **Censoring:** Active loans at data cutoff are censored — we know they survived *at least* N months.
- **Log-rank test:** Tests if survival curves from different groups are statistically different.

**Credit risk application:** KM confirms not just *if* but *when* default occurs. IFRS 9 lifetime PD estimation relies on this logic.

In [ ]:
!pip install lifelines --quiet

from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test

# Prepare survival dataset
survival_prep = df[['last_pymnt_d', 'issue_d']].copy()
survival_prep['last_pymnt_d'] = pd.to_datetime(survival_prep['last_pymnt_d'], format='mixed')
survival_prep['issue_d']      = pd.to_datetime(survival_prep['issue_d'],      format='mixed')
survival_prep['duration_months'] = (
    (survival_prep['last_pymnt_d'].dt.year  - survival_prep['issue_d'].dt.year)  * 12 +
    (survival_prep['last_pymnt_d'].dt.month - survival_prep['issue_d'].dt.month)
).clip(lower=1)

survival_df = df_cleaned[['default', 'fico_range_low', 'grade']].join(
    survival_prep['duration_months'], how='inner')
survival_df['purpose'] = df['purpose'].reindex(df_cleaned.index)
survival_df = survival_df.dropna()
print(f"Survival dataset: {survival_df.shape}, default rate: {survival_df['default'].mean():.4f}")

# KM by Grade
fig, ax = plt.subplots(figsize=(12, 7))
kmf = KaplanMeierFitter()
grade_colors = {'A':'#2ecc71','B':'#27ae60','C':'#f39c12','D':'#e67e22','E':'#e74c3c','F':'#c0392b','G':'#8e44ad'}
for grade, color in grade_colors.items():
    mask = survival_df['grade'] == grade
    if mask.sum() > 100:
        kmf.fit(survival_df.loc[mask,'duration_months'], survival_df.loc[mask,'default'],
                label=f'Grade {grade} (n={mask.sum():,})')
        kmf.plot_survival_function(ax=ax, color=color, ci_show=False)
ax.set_title('Kaplan-Meier Survival Curves by Loan Grade')
ax.set_xlabel('Months Since Origination')
ax.set_ylabel('Survival Probability')
ax.legend(loc='lower left')
ax.set_ylim(0.5, 1.0)
plt.tight_layout()
plt.show()

# KM by FICO Band
survival_df['fico_band'] = pd.cut(survival_df['fico_range_low'],
    bins=[0,650,670,690,720,750,850], labels=['<650','650-670','670-690','690-720','720-750','750+'])
fig, ax = plt.subplots(figsize=(12, 7))
for band, color in zip(['<650','650-670','670-690','690-720','720-750','750+'],
                        ['#e74c3c','#e67e22','#f39c12','#2ecc71','#27ae60','#1abc9c']):
    mask = survival_df['fico_band'] == band
    if mask.sum() > 100:
        kmf.fit(survival_df.loc[mask,'duration_months'], survival_df.loc[mask,'default'],
                label=f'FICO {band}')
        kmf.plot_survival_function(ax=ax, color=color, ci_show=False)
ax.set_title('Kaplan-Meier Survival Curves by FICO Band')
ax.set_ylim(0.5, 1.0)
ax.legend(loc='lower left')
plt.tight_layout()
plt.show()

# Multivariate log-rank test
print("\n--- Log-Rank Test: Are Grade Curves Different? ---")
results = multivariate_logrank_test(
    event_durations=survival_df['duration_months'],
    groups=survival_df['grade'],
    event_observed=survival_df['default']
)
results.print_summary()
print(f"P-value: {results.p_value:.6f}")

**Result:** Grade G reaches 50% cumulative default by month 20. Grade A survives to 82% at month 60. Log-rank p≈0 confirms differences are not random.